# Bronze Layer

## Step -1 Define Schema as STRING

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

## Step -2 Read Raw as CSV

In [0]:
orders_bronze_df = spark.read.format('csv').option("header","true").schema(orders_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")

customers_bronze_df = spark.read.format('csv').option("header", "true").schema(customer_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")

In [0]:
#Display the csv files
orders_bronze_df.display()
customers_bronze_df.display()

## Step - 3 Add Ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp
#Add a new column 'ingest_time' to the dataframe
orders_bronze_df = orders_bronze_df.withColumn('ingest_time', current_timestamp())
customers_bronze_df = customers_bronze_df.withColumn('ingest_time', current_timestamp())


In [0]:
#Display the csv files we can see the ingested time stamp and last column
orders_bronze_df.display()
customers_bronze_df.display()

## Step -4 Save AS Delta Table in Bronze

In [0]:
#Write the dataframe to delta lake
orders_bronze_df.write.format('delta').mode("overwrite").saveAsTable('retail_project.bronze.bronze_orders')
customers_bronze_df.write.format('delta').mode("overwrite").saveAsTable('retail_project.bronze.bronze_customers')

In [0]:
#Structure of Bronze Layer
# Retail Voulume
# │
# ├── raw_file/
# │   ├── retail_orders/
# │   └── retail_customers/
# │
# ├── bronze/ (schema)
# │   ├── bronze_orders (orders_table)
# │   └── bronze_customers (customers_table)

# Silver Layer Transformation

#### 🎯 OBJECTIVE to check on Silver Layer

In this lecture, we will clean the retail orders and customer dataset by:

- Fixing date formats

- Standardizing category values

- Removing duplicates

- Handling null values

- Converting discount from "5%" to 0.05 and "10%" to 0.1

- Removing negative quantities

- Fixing casing issues

- Casting unit_price to decimal

- Validating emails

- Recalculating total_amount

- and more

#### STEP 1: Load Bronze Orders

In [0]:
bronze_order_df = spark.read.table('retail_project.bronze.bronze_orders')

display(bronze_order_df)

#### 👨‍🏫 STEP 2: Fix Date Formats

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
        coalesce(
            try_to_date('order_date', 'yyyy-MM-dd'),
            try_to_date('order_date', 'dd-MM-yyyy'),
            try_to_date('order_date', 'yyyy/MM/dd'),
            try_to_date('order_date', 'dd/MM/yyyy')
        )
)

silver_df.display()

#### 👨‍🏫 STEP 3: Standardize Category Values

In [0]:
#Select the category column and display the distinct values
silver_df.select('category').distinct().display()

Here, we can clearly observe that the same category appears in different formats due to variations in uppercase and lowercase letters, special characters, and spelling mistakes. These values should be standardized into a single, consistent category.

For example:

* **Electronic**, **electronics**, and **Electronics** should be standardized as **Electronics**.
* **Fashon** and **Fashion** should be standardized as **Fashion**.
* **Home & Living** and **Home and Living** should be standardized as **Home & Living**.

Creating a standardized category column ensures consistency in the data, improves data quality, and enables accurate analysis and reporting.


In [0]:
from pyspark.sql.functions import upper, trim, when

silver_df = silver_df.withColumn(
    'category',
    upper(trim(col('category')))
)
silver_df.select('category').distinct().display() 
#Here below we converted into Upper case

In [0]:
# Now we can Reassign the standardized name for the category

silver_df = silver_df.withColumn(
    'category',
    when(col('category') == 'ELECTRONIC','ELECTRONICS').
    when(col('category') == 'FASHON', 'FASHION').
    when(col('category') == 'HOME AND LIVING','HOME & LIVING').
    otherwise(col('category'))
)
#Now we can see the standardized name for the category
silver_df.select('category').distinct().display()


#### STEP 4: Remove Duplicate Orders

In [0]:
from pyspark.sql.functions import count, countDistinct

#Checking the number of duplicate records
silver_df.select(count('order_id'), countDistinct('order_id')).display()


In [0]:
#Remove the duplicates
silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'), countDistinct('order_id')).display()


#### 👨‍🏫 STEP 5: Handle Null Values discount

In [0]:
silver_df.select('discount').distinct().display()

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn(
    'discount',
    when(col('discount').isNull(), '0').
    when(col('discount')=='ten', '0.1').
    otherwise(col('discount'))
)

In [0]:
silver_df.display()

#### 👨‍🏫 STEP 6: Convert Discount from "5%" and "10%" to Decimal.

some rows contain:

5% and 10%

we need change 0.05 and 0.1 respectively.


In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    'discount',
    when(col('discount').like('5%'), (regexp_replace('discount', '5%', '0.05'))).
    when(col('discount').like('10%'), (regexp_replace('discount', '10%', '0.1'))).
    otherwise(col('discount'))
)

silver_df.display()

In [0]:
silver_df.select('discount').distinct().display()

#### 👨‍🏫 STEP 7: Remove Negative Quantity

- Here, some of the rows in quantity contains negative values, quantity never been in negative its ranges from 0 to infinity.

In [0]:
# need to filter the quantity >=0

silver_df = silver_df.filter(col('quantity') >= 0)

silver_df.display()

In [0]:
silver_df.select(count('order_id'), countDistinct('order_id')).display()
# Here we can see from 10000 rows to 7487 rows after removing the negative quantity

#### 👨‍🏫 STEP 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn(
    'unit_price',
    col('unit_price').try_cast(DecimalType(10, 2))
)

silver_df.display()

#### 👨‍🏫 STEP 9: Recalculate Total Amount

In [0]:
silver_df = silver_df.withColumn(
    'total_amount',
    col('quantity').cast('int') * col('unit_price') * (1 - col('discount').cast('double'))
)

silver_df.display()

#### 👨‍🏫 STEP 10: Save as Silver Table

In [0]:
silver_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.silver.silver_orders')

### Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project

#### 👨‍🏫 STEP 1: Read Bronze Table

In [0]:
customers_bronze_df = spark.read.table('retail_project.bronze.bronze_customers')

display(customers_bronze_df)

#### 👨‍🏫 STEP 2: Trim Spaces

In [0]:
from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_time")
)

display(customers_df)

#### 👨‍🏫 STEP 3: Remove Duplicate Customers

In [0]:
customers_df.select(count('customer_id'), countDistinct('customer_id')).display()

- Here, we can observe the 20 Duplicate customers so, we can drop those duplicate customer by keeping lastest one with help of ingest_time remove the old one.


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy('customer_id').orderBy(desc('ingest_time'))

customers_df = customers_df.withColumn(
    'row_num', row_number().over(window_spec)
)

customers_df = customers_df.filter('row_num = 1').drop('row_num')

customers_df.display()

In [0]:
customers_df.select(count('customer_id'), countDistinct('customer_id')).display()

#### 👨‍🏫 STEP 4: Fix and Standardize Status

In [0]:
customers_df.select(['status']).distinct().display()

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    'status',upper(col('status'))
)

customers_df = customers_df.withColumn(
    'status', 
    when(col('status').isin('ACTIVE', 'INACTIVE'),  col('status')).otherwise('UNKNOWN')
)
customers_df.display()

In [0]:
customers_df.select(['status']).distinct().display()

#### 👨‍🏫 STEP 5: Convert loyalty_points to Integer

- Convert non interger values into null values


In [0]:
from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    'loyalty_points',
    regexp_replace(col('loyalty_points'), '[^0-9]', '')
)

customers_df = customers_df.withColumn(
    'loyalty_points',
    col('loyalty_points').try_cast('int')
)

customers_df.display()

#### 👨‍🏫 STEP 6: Validate Email Format

In [0]:
customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

# Example: meenadas@email.com - valid, karansingh@email - invalid

# customers_df.select("email", "email_valid").display()

customers_df = customers_df.filter(col("email_valid") == True)

#### 👨‍🏫 STEP 7: Fix Date Formats

In [0]:
#Here we created new column dob_parsed and registration_parsed using date_of_birth and registration_date columns and used coalesce function to parse the date in different formats
from pyspark.sql.functions import to_date, coalesce

customers_df = customers_df.withColumn(
    "dob_parsed", 
    coalesce(
            try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
            try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
            try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
            try_to_date(col('date_of_birth'),'dd/MM/yyyy')
        )
)

customers_df = customers_df.withColumn(
    "registration_parsed",
    coalesce(
            try_to_date(col('registration_date'),'yyyy-MM-dd'),
            try_to_date(col('registration_date'),'dd-MM-yyyy'),
            try_to_date(col('registration_date'),'yyyy/MM/dd'),
            try_to_date(col('registration_date'),'dd/MM/yyyy')
        )
)

customers_df.display()

In [0]:
#Here we droped the old columns date_of_birth and registration_date and renamed the new columns dob_parsed and registration_parsed to date_of_birth and registration_date respectively

customers_df = customers_df.drop('date_of_birth','registration_date')\
    .withColumnRenamed('dob_parsed','date_of_birth')\
    .withColumnRenamed('registration_parsed','registration_date')

customers_df.display()

#### 👨‍🏫 STEP 8: Standardize Gender

In [0]:
customers_df.select(['gender']).distinct().display()

In [0]:
customers_df = customers_df.withColumn(
    'gender',
    upper(col('gender'))
)

customers_df = customers_df.withColumn(
    'gender',
    when(col('gender') == 'MALE', 'M')
        .when(col('gender') == 'FEMALE', 'F')
        .otherwise('U')
)
customers_df.display()

In [0]:
customers_df.select(['gender']).distinct().display()

#### 👨‍🏫 STEP 9: Save as Customers Table in Silver Layer


In [0]:
customers_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.silver.silver_customers')

In [0]:
customers_df.select(['phone']).distinct().display()

In [0]:
customers_df.select(count('customer_id'), countDistinct('customer_id')).display()

### Gold Layer in Databricks – Building Fact & Dimension Tables | Retail End-to-End Project

#### 🎯 OBJECTIVE OF THIS LECTURE
We will:

1️⃣**Create Dimension Tables**

2️⃣ **Create Fact Table**

3️⃣ **Join Orders & Customers**

4️⃣ **Build KPI-ready aggregates**

5️⃣ **Store optimized Gold tables**

#### 👨‍🏫 STEP 1: Load Silver Tables

In [0]:
customers_df = spark.read.table('retail_project.silver.silver_customers')
orders_df = spark.read.table('retail_project.silver.silver_orders')

customers_df.display()
orders_df.display()

#### 👨‍🏫 STEP 2: Create Customer Dimension

In [0]:
#Based on the Business case we can select the specific columns from the silver tables(customers_df)
dim_customers = customers_df.select('customer_id',
                                    'customer_name',
                                    'email',
                                    'city',
                                    'state',
                                    'loyalty_points',
                                    'status')

dim_customers.display()

#### 👨‍🏫 STEP 2.1: Save as Table in Gold Layer for dim_customers

In [0]:
dim_customers.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_dim_customers')

#### 👨‍🏫 STEP 3: Create Fact Sales Table

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    'left'
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

fact_sales.display()

In [0]:
#Here we extracted the Year, Month and date from the order_date column in fact_sales to create new columns Year, Month and Day
from pyspark.sql.functions import year, month, dayofmonth

orders_enriched = fact_sales.\
    withColumn('Year', year('order_date')).\
    withColumn('Month', month('order_date')).\
    withColumn('Day', dayofmonth('order_date'))

orders_enriched.display()



In [0]:
#Now we build to create the gold table for business related KPI's

from pyspark.sql.functions import sum, avg, count, min, max, countDistinct

gold_df = orders_enriched.groupBy(
    'product_id',
    'product_name',
    'customer_id',
    'customer_name',
    'category',
    'Year',
    'Month',
    'order_date',
    'gender',
    'city',
    'state',
    'loyalty_points'
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)

gold_df.display()


#### 👨‍🏫 STEP 3.1: Save as Table in Gold Layer for fact_sales


In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.gold_fact_sales')

In [0]:
gold_layer = spark.read.table('retail_project.gold.gold_fact_sales')
display(gold_layer)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Count null values in each column
null_counts = gold_layer.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in gold_layer.columns
])

null_counts.display()